# PyTorch: Multi-Class Classification using FastAI

Dataset: PETS

In [ ]:
import re
from PIL import Image

In [ ]:
from fastai.vision.all import *

# Prepare Dataset

In [ ]:
path = untar_data(URLs.PETS)

In [ ]:
pets = DataBlock(
    blocks=(ImageBlock, CategoryBlock),
    get_items=get_image_files,
    splitter=RandomSplitter(seed=13),
    get_y=using_attr(RegexLabeller(r"(.+)_\d+.jpg$"), "name"),
    # Apply presizing approach
    item_tfms=Resize(460),
    batch_tfms=aug_transforms(size=224, min_scale=0.75)
)

In [ ]:
# Summarize all transformations
pets.summary(path/"images")

In [ ]:
dls = pets.dataloaders(path/"images")

In [ ]:
dls.show_batch(nrows=1, ncols=3)

# Train Model

## Fine Tuning (Auto)

In [ ]:
learn = vision_learner(dls, resnet34, metrics=error_rate)

In [ ]:
# Try to find learning rate (somewhere between lr_steep and lr_min)
lr_min, lr_steep = learn.lr_find(suggest_funcs=(minimum, steep))
print(f"Minimum/10: {lr_min:.2e}, steepest point: {lr_steep:.2e}")

In [ ]:
learn.fine_tune(6, base_lr=3e-3, freeze_epochs=3)

In [ ]:
learn.recorder.plot_loss()

## Fine Tuning (Manual)

In [ ]:
learn = vision_learner(dls, resnet34, metrics=error_rate)

In [ ]:
# Try to find learning rate (somewhere between lr_steep and lr_min)
lr_min, lr_steep = learn.lr_find(suggest_funcs=(minimum, steep))
print(f"Minimum/10: {lr_min:.2e}, steepest point: {lr_steep:.2e}")

In [ ]:
# Train randomly added layers for three epochs
learn.fit_one_cycle(3, 3e-3)

In [ ]:
# Unfreeze pre-trained layers
learn.unfreeze()

In [ ]:
# Find LR again (previously found LR isn't appropriate any more)
learn.lr_find()

In [ ]:
learn.fit_one_cycle(6, lr_max=1e-5)

## Fine Tuning (Discriminative LR)
(The most confident way to fine tune a pre-trained model)

In [ ]:
learn = vision_learner(dls, resnet34, metrics=error_rate)

In [ ]:
# Try to find learning rate (somewhere between lr_steep and lr_min)
lr_min, lr_steep = learn.lr_find(suggest_funcs=(minimum, steep))
print(f"Minimum/10: {lr_min:.2e}, steepest point: {lr_steep:.2e}")

In [ ]:
# Train randomly added layers for three epochs
learn.fit_one_cycle(3, 3e-3)

In [ ]:
# Unfreeze pre-trained layers
learn.unfreeze()

In [ ]:
learn.fit_one_cycle(12, lr_max=slice(1e-6,1e-4))

In [ ]:
learn.recorder.plot_loss()

# Evaluate

In [ ]:
xb, yb = dls.one_batch()
yb_hat, _ = learn.get_preds(dl=[(xb, yb)])
xb.shape, yb.shape,yb_hat.shape,yb_hat[0].sum()

In [ ]:
interp = ClassificationInterpretation.from_learner(learn)
interp.plot_confusion_matrix(figsize=(12,12), dpi=60)

In [ ]:
interp.most_confused(min_val=5)